# HuggingFace Transformers: API, Models and Fine-Tuning Techniques

## Agenda for Day 1 

#### Module 1: Introduction to Transformers and the HuggingFace Ecosystem (90 mins)
- **The Evolution of NLP: From RNNs to Transformers**
  - The attention mechanism revolution
  - The "Attention Is All You Need" paper in a nutshell
  - Encoder-decoder architecture overview
  - Pre-training and fine-tuning paradigm

- **HuggingFace Ecosystem Overview**
  - The HuggingFace Hub: Models, Datasets, and Spaces
  - Core libraries: `transformers`, `datasets`, `tokenizers`, `accelerate`
  - Model cards, dataset cards, and community standards
  - Navigating the Hub: finding models for your use case

- **Hands-on Lab (45 mins)**
  - Setting up the environment and installing dependencies
  - Loading your first pre-trained model
  - Exploring model configs and architectures programmatically
  - Browsing and downloading datasets from the Hub

#### Module 2: The Transformers API Foundation (75 mins)
- **The Pipeline API: Zero-Shot Inference**
  - Understanding the high-level pipeline abstraction
  - Default models and task-specific pipelines
  - Batch processing with pipelines

- **Tokenizer Fundamentals**
  - Tokenization strategies: BPE, WordPiece, SentencePiece
  - The tokenizer-model relationship
  - Padding, truncation, and attention masks
  - Special tokens and their significance

- **Hands-on Lab (30 mins)**
  - Running inference with pipelines (sentiment analysis, NER, summarization)
  - Tokenization deep dive: encoding/decoding text
  - Comparing tokenizers across model architectures
  - Handling variable-length sequences and batching

---

### Setup environment for HuggingFace

```bash
conda create -n hft_env python=3.14 -y
conda activate hft_env
conda install python pip jupyter
pip install dotenv
pip install torch
pip install transformers datasets evaluate accelerate timm
```

#### Installation (using standard virtualenv + pip)
```bash
python -m venv hft_env
source hft_env/bin/activate
pip install jupyter dotenv
pip install torch
pip install transformers datasets evaluate accelerate timm
``` 

---

### Verify installation

In [7]:
import transformers
print(transformers.__version__)

5.14.1


In [4]:
import torch
print(torch.__version__)
print(torch.accelerator.is_available())
print(torch.accelerator.current_accelerator())

2.13.0
True
mps


---

### Sign in to HuggingFace

In [8]:
from dotenv import load_dotenv
import os

load_dotenv("../dotenv.sh")
HF_TOKEN = os.getenv("HF_TOKEN")

from huggingface_hub import login
login(token=HF_TOKEN)

---

### Explore HuggingFace Hub Programmatically


In [ ]:
from huggingface_hub import list_models

# List top 10 models sorted by most downloads
models = list_models(sort="downloads", limit=10)
for model in models:
    print(f"Model: {model.modelId}, Downloads: {model.downloads:,}")

Model: sentence-transformers/all-MiniLM-L6-v2, Downloads: 248,516,777
Model: cross-encoder/ms-marco-MiniLM-L6-v2, Downloads: 83,870,749
Model: google-bert/bert-base-uncased, Downloads: 78,430,360
Model: BAAI/bge-small-en-v1.5, Downloads: 65,211,911
Model: google/electra-base-discriminator, Downloads: 54,637,196
Model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2, Downloads: 48,923,744
Model: BAAI/bge-m3, Downloads: 34,670,275
Model: sentence-transformers/all-mpnet-base-v2, Downloads: 29,737,160
Model: google-t5/t5-small, Downloads: 27,245,112
Model: Qwen/Qwen3-0.6B, Downloads: 26,124,974


In [11]:
from huggingface_hub import list_models

g = list_models()
g


<generator object HfApi.list_models at 0x17f019380>

In [32]:
list_models?

Signature:
list_models(
    *,
    filter: 'str | Iterable[str] | None' = None,
    author: 'str | None' = None,
    apps: 'str | list[str] | None' = None,
    gated: 'bool | None' = None,
    inference: "Literal['warm'] | None" = None,
    inference_provider: "Literal['all'] | PROVIDER_T | list[PROVIDER_T] | None" = None,
    model_name: 'str | None' = None,
    trained_dataset: 'str | list[str] | None' = None,
    search: 'str | None' = None,
    pipeline_tag: 'str | None' = None,
    num_parameters: 'str | None' = None,
    emissions_thresholds: 'tuple[float, float] | None' = None,
    sort: 'ModelSort_T | None' = None,
    limit: 'int | None' = None,
    expand: 'list[ExpandModelProperty_T] | None' = None,
    full: 'bool | None' = None,
    cardData: 'bool' = False,
    fetch_config: 'bool' = False,
    token: 'bool | str | None' = None,
) -> 'Iterable[ModelInfo]'
Docstring:
List models hosted on the Huggingface Hub, given some filters.

Args:
    filter (`str` or `Iterable[str]`,

In [33]:
m = next(g)

In [34]:
m

ModelInfo(id='Cactus-Compute/needle', author=None, base_models=None, card_data=None, children_model_count=None, config=None, created_at=datetime.datetime(2026, 3, 16, 4, 1, 31, tzinfo=datetime.timezone.utc), disabled=None, downloads=955, downloads_all_time=None, eval_results=None, gated=None, gguf=None, inference=None, inference_provider_mapping=None, last_modified=None, library_name='jax', likes=280, mask_token=None, model_index=None, pipeline_tag=None, private=False, resource_group=None, safetensors=None, security_repo_status=None, sha=None, siblings=None, spaces=None, tags=['jax', 'safetensors', 'needle', 'function-calling', 'tool-use', 'encoder-decoder', 'edge', 'on-device', 'flax', 'license:mit', 'region:us'], transformers_info=None, trending_score=99, used_storage=None, widget_data=None)

In [37]:
# Find models for a specific task
from huggingface_hub import HfApi
api = HfApi()
sentiment_models = list(api.list_models(
   filter="text-classification", 
   sort="downloads", 
   limit=5
))

for model in sentiment_models:
    print(f"Model: {model.modelId}, Downloads: {model.downloads:,}")

Model: cross-encoder/ms-marco-MiniLM-L6-v2, Downloads: 83,870,749
Model: BAAI/bge-reranker-v2-m3, Downloads: 17,535,094
Model: cross-encoder/ms-marco-MiniLM-L4-v2, Downloads: 10,844,478
Model: ProsusAI/finbert, Downloads: 7,356,134
Model: BAAI/bge-reranker-base, Downloads: 4,775,142


In [40]:
m = sentiment_models[0]
m

ModelInfo(id='cross-encoder/ms-marco-MiniLM-L6-v2', author=None, base_models=None, card_data=None, children_model_count=None, config=None, created_at=datetime.datetime(2022, 3, 2, 23, 29, 5, tzinfo=datetime.timezone.utc), disabled=None, downloads=83870749, downloads_all_time=None, eval_results=None, gated=None, gguf=None, inference=None, inference_provider_mapping=None, last_modified=None, library_name='sentence-transformers', likes=285, mask_token=None, model_index=None, pipeline_tag='text-ranking', private=False, resource_group=None, safetensors=None, security_repo_status=None, sha=None, siblings=None, spaces=None, tags=['sentence-transformers', 'pytorch', 'jax', 'onnx', 'safetensors', 'openvino', 'bert', 'text-classification', 'transformers', 'text-ranking', 'en', 'dataset:sentence-transformers/msmarco', 'base_model:cross-encoder/ms-marco-MiniLM-L12-v2', 'base_model:quantized:cross-encoder/ms-marco-MiniLM-L12-v2', 'license:apache-2.0', 'text-embeddings-inference', 'endpoints_compati

In [45]:
m.gated

In [51]:
sm = list_models(filter="llama", sort="downloads", limit=5, gated=False)
m = next(sm)
m

ModelInfo(id='hmellor/tiny-random-LlamaForCausalLM', author=None, base_models=None, card_data=None, children_model_count=None, config=None, created_at=datetime.datetime(2025, 4, 29, 21, 47, 13, tzinfo=datetime.timezone.utc), disabled=None, downloads=5513574, downloads_all_time=None, eval_results=None, gated=None, gguf=None, inference=None, inference_provider_mapping=None, last_modified=None, library_name='transformers', likes=0, mask_token=None, model_index=None, pipeline_tag='text-generation', private=False, resource_group=None, safetensors=None, security_repo_status=None, sha=None, siblings=None, spaces=None, tags=['transformers', 'safetensors', 'llama', 'text-generation', 'conversational', 'arxiv:1910.09700', 'text-generation-inference', 'endpoints_compatible', 'region:us'], transformers_info=None, trending_score=None, used_storage=None, widget_data=None)

---

### Load and inspect a model


In [52]:
from transformers import AutoModel, AutoConfig, AutoTokenizer

model_name = "bert-base-uncased"
config = AutoConfig.from_pretrained(model_name)
print(f"Model type: {config.model_type}")
print(f"Number of hidden layers: {config.num_hidden_layers}")
print(f"Hidden size: {config.hidden_size}")
print(f"Number of attention heads: {config.num_attention_heads}")
print(f"Vocabulary size: {config.vocab_size}")

Model type: bert
Number of hidden layers: 12
Hidden size: 768
Number of attention heads: 12
Vocabulary size: 30522


In [53]:
config

BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.14.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

In [55]:
# Silence warning about UNEXPECTED keys
import logging

logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

In [56]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
print(f"Tokenizer: {tokenizer}")
print(f"Model: {model}")

[transformers] The following layers were not sharded: encoder.layer.*.output.LayerNorm.bias, encoder.layer.*.attention.output.dense.bias, encoder.layer.*.attention.self.query.bias, encoder.layer.*.attention.self.key.weight, pooler.dense.bias, embeddings.word_embeddings.weight, encoder.layer.*.attention.self.value.bias, encoder.layer.*.attention.output.dense.weight, encoder.layer.*.output.LayerNorm.weight, embeddings.LayerNorm.weight, encoder.layer.*.intermediate.dense.bias, encoder.layer.*.attention.output.LayerNorm.bias, encoder.layer.*.output.dense.bias, encoder.layer.*.output.dense.weight, embeddings.token_type_embeddings.weight, encoder.layer.*.attention.self.query.weight, pooler.dense.weight, encoder.layer.*.attention.output.LayerNorm.weight, encoder.layer.*.intermediate.dense.weight, embeddings.position_embeddings.weight, encoder.layer.*.attention.self.value.weight, encoder.layer.*.attention.self.key.bias, embeddings.LayerNorm.bias


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Tokenizer: BertTokenizer(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})
Model: BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_ty

In [57]:

total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters in the model: {total_params:,}")

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of trainable parameters in the model: {trainable_params:,}")


Total number of parameters in the model: 109,482,240
Number of trainable parameters in the model: 109,482,240


---

### Tokenizer Deep Dive


In [1]:
from transformers import AutoTokenizer
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer

BertTokenizer(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [3]:
#text = "The transformer model is revolutionizing NLP!"

text = "Hi, How are you doing today? I hope you're having a great day!"

# Tokenize -> Encode -> Decode
tokens = tokenizer.tokenize(text)
print(f"Tokens: {tokens}")


Tokens: ['hi', ',', 'how', 'are', 'you', 'doing', 'today', '?', 'i', 'hope', 'you', "'", 're', 'having', 'a', 'great', 'day', '!']


In [6]:
text = "Python is an easy language for ML workflow. Python is freely downloadable."
print(tokenizer(text))
print(tokenizer.tokenize(text))


{'input_ids': [101, 18750, 2003, 2019, 3733, 2653, 2005, 19875, 2147, 12314, 1012, 18750, 2003, 10350, 26720, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
['python', 'is', 'an', 'easy', 'language', 'for', 'ml', 'work', '##flow', '.', 'python', 'is', 'freely', 'downloadable', '.']


In [7]:

encoded = tokenizer(text)
print(f"Encoded input IDs: {encoded['input_ids']}")
print(f"Attn Mask: {encoded['attention_mask']}")

decoded = tokenizer.decode(encoded['input_ids'])
print(f"Decoded text: {decoded}")

Encoded input IDs: [101, 18750, 2003, 2019, 3733, 2653, 2005, 19875, 2147, 12314, 1012, 18750, 2003, 10350, 26720, 1012, 102]
Attn Mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Decoded text: [CLS] python is an easy language for ml workflow. python is freely downloadable. [SEP]


In [22]:
# Batch encoding with padding
texts = [
    "Transformers are amazing but complex. Python makes it easy.",
    "Python is cool."
] 

batch_encoded = tokenizer(texts, padding='max_length', max_length=25)
print(batch_encoded)
#print(f"\nBatch shape: {batch_encoded['input_ids'].shape}")
#print(f"Padded IDs:\n{batch_encoded['input_ids']}")

{'input_ids': [[101, 19081, 2024, 6429, 2021, 3375, 1012, 18750, 3084, 2009, 3733, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [101, 18750, 2003, 4658, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}


In [8]:
tokenizer?

Signature:     
tokenizer(
    text: 'TextInput | PreTokenizedInput | list[TextInput] | list[PreTokenizedInput] | None' = None,
    text_pair: 'TextInput | PreTokenizedInput | list[TextInput] | list[PreTokenizedInput] | None' = None,
    text_target: 'TextInput | PreTokenizedInput | list[TextInput] | list[PreTokenizedInput] | None' = None,
    text_pair_target: 'TextInput | PreTokenizedInput | list[TextInput] | list[PreTokenizedInput] | None' = None,
    add_special_tokens: 'bool' = True,
    padding: 'bool | str | PaddingStrategy' = False,
    truncation: 'bool | str | TruncationStrategy | None' = None,
    max_length: 'int | None' = None,
    stride: 'int' = 0,
    is_split_into_words: 'bool' = False,
    pad_to_multiple_of: 'int | None' = None,
    padding_side: 'str | None' = None,
    return_tensors: 'str | TensorType | None' = None,
    return_token_type_ids: 'bool | None' = None,
    return_attention_mask: 'bool | None' = None,
    return_overflowing_tokens: 'bool' = False,
    

---

### Download and Explore a Dataset

In [24]:
from datasets import load_dataset

# Load IMDB dataset for sentiment analysis
dataset = load_dataset("stanfordnlp/imdb")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [31]:
for d in dataset["train"]:
    if d["label"] == 1:
        print(f"Positive review: {d['text']}")
        break

Positive review: Zentropa has much in common with The Third Man, another noir-like film set among the rubble of postwar Europe. Like TTM, there is much inventive camera work. There is an innocent American who gets emotionally involved with a woman he doesn't really understand, and whose naivety is all the more striking in contrast with the natives.<br /><br />But I'd have to say that The Third Man has a more well-crafted storyline. Zentropa is a bit disjointed in this respect. Perhaps this is intentional: it is presented as a dream/nightmare, and making it too coherent would spoil the effect. <br /><br />This movie is unrelentingly grim--"noir" in more than one sense; one never sees the sun shine. Grim, but intriguing, and frightening.


In [32]:

print(f"Dataset splits: {dataset.keys()}")
print(f"Train size: {len(dataset['train'])}")
print(f"Test size: {len(dataset['test'])}")

# Explore a sample
sample = dataset['train'][0]
print(f"\nSample text: {sample['text'][:200]}...")
print(f"Label: {sample['label']} ({'Positive' if sample['label'] == 1 else 'Negative'})")

# Check label distribution
from collections import Counter
labels = dataset['train']['label']
print(f"\nLabel distribution: {Counter(labels)}")

# Load a specific subset (faster for experimentation)
small_train = dataset['train'].select(range(1000))
small_test = dataset['test'].select(range(500))
print(f"\nSmall train size: {len(small_train)}")
print(f"Small test size: {len(small_test)}")

Dataset splits: dict_keys(['train', 'test', 'unsupervised'])
Train size: 25000
Test size: 25000

Sample text: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ev...
Label: 0 (Negative)

Label distribution: Counter({0: 12500, 1: 12500})

Small train size: 1000
Small test size: 500


---

### Using the ```pipeline()``` from the ```transformers``` library

In [2]:
from transformers import pipeline
classifier = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
result = classifier("I love machine learning!")
print(result)

result = classifier("Transformers are amazing but complex.")
print(result)

result = classifier("I hate bugs in my code.")
print(result)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9998431205749512}]
[{'label': 'POSITIVE', 'score': 0.9989979863166809}]
[{'label': 'NEGATIVE', 'score': 0.999018669128418}]


---

### Pipeline Examples: NLP Tasks


In [31]:
# 1. Sentiment Analysis
classifier = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
results = classifier(["Great product!", "Terrible experience..."])
# [{'label': 'POSITIVE', 'score': 0.99}, {'label': 'NEGATIVE', 'score': 0.98}]
print(results)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9998729228973389}, {'label': 'NEGATIVE', 'score': 0.9996969699859619}]


In [34]:
# Silencing warnings for cleaner output
import logging
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

In [35]:
# 2. Named Entity Recognition

ner = pipeline("ner", aggregation_strategy="simple", model="dbmdz/bert-large-cased-finetuned-conll03-english")
text = "Elon Musk founded SpaceX in Hawthorne, California."
entities = ner(text)
print(entities)
for entity in entities:
    print(f"Entity: {entity['word']}, Type: {entity['entity_group']}, Score: {entity['score']:.4f}") 

[transformers] The following layers were not sharded: bert.encoder.layer.*.attention.output.dense.bias, bert.encoder.layer.*.intermediate.dense.weight, bert.embeddings.token_type_embeddings.weight, bert.encoder.layer.*.intermediate.dense.bias, bert.encoder.layer.*.output.LayerNorm.bias, classifier.weight, bert.encoder.layer.*.output.dense.bias, bert.embeddings.LayerNorm.bias, classifier.bias, bert.embeddings.word_embeddings.weight, bert.embeddings.LayerNorm.weight, bert.encoder.layer.*.attention.output.LayerNorm.bias, bert.encoder.layer.*.output.dense.weight, bert.encoder.layer.*.attention.self.key.weight, bert.encoder.layer.*.attention.self.query.weight, bert.encoder.layer.*.output.LayerNorm.weight, bert.encoder.layer.*.attention.self.value.weight, bert.encoder.layer.*.attention.self.query.bias, bert.embeddings.position_embeddings.weight, bert.encoder.layer.*.attention.self.value.bias, bert.encoder.layer.*.attention.output.LayerNorm.weight, bert.encoder.layer.*.attention.output.dense.

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[{'entity_group': 'PER', 'score': np.float32(0.9986373), 'word': 'Elon Musk', 'start': 0, 'end': 9}, {'entity_group': 'ORG', 'score': np.float32(0.9988726), 'word': 'SpaceX', 'start': 18, 'end': 24}, {'entity_group': 'LOC', 'score': np.float32(0.99004865), 'word': 'Hawthorne', 'start': 28, 'end': 37}, {'entity_group': 'LOC', 'score': np.float32(0.9986236), 'word': 'California', 'start': 39, 'end': 49}]
Entity: Elon Musk, Type: PER, Score: 0.9986
Entity: SpaceX, Type: ORG, Score: 0.9989
Entity: Hawthorne, Type: LOC, Score: 0.9900
Entity: California, Type: LOC, Score: 0.9986


In [ ]:
# 3. Question-Answering

qa = pipeline("question-answering", model='distilbert/distilbert-base-cased-distilled-squad')

result = qa(
    question="What is the capital of France?",
    context="France is a country in Western Europe. Its capital is Paris."
)
print(result)


In [ ]:
# 4. Summarization
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
text = """The transformer model, introduced in the paper 'Attention Is All You Need',
has revolutionized the field of natural language processing. Unlike previous 
sequence-to-sequence models that relied on recurrent neural networks, the 
transformer uses self-attention mechanisms to process entire sequences 
simultaneously, enabling better parallelization and capturing long-range 
dependencies more effectively."""

summary = summarizer(text, max_length=50, min_length=25)
print(f"Summary: {summary[0]['summary_text']}")